# CC-CCII COVID-19 Dataset - Download to Google Drive

**Source:** https://download.cncb.ac.cn/covid-ct/  (public, không cần đăng ký)

**Kết quả:**
```
Drive/thesis/data/covid/
  images/           ← CT slices có tổn thương (PNG)
  lesions_slices.csv
  metadata.csv
```

**Thời gian ước tính:** ~30–60 phút (31 × ~150 MB = ~4.5 GB tổng)

> Chỉ tải **COVID19** zips (ca bệnh). CP / Normal không cần cho segmentation.

In [ ]:
# ── Cấu hình ──────────────────────────────────────────────────────────

# Thư mục lưu trên Google Drive
DRIVE_OUTPUT = "/content/drive/MyDrive/thesis/data/covid"

# Tải toàn bộ 31 zip (True) hoặc chỉ một vài zip để test (False)
DOWNLOAD_ALL = True

# Nếu DOWNLOAD_ALL = False => chỉ tải các zip này (để test nhanh)
TEST_PARTS = [1, 2, 3]

# Lọc slice: chỉ giữ slice được liệt kê trong lesions_slices.csv
# True  => chỉ CT slice có tổn thương (~vài trăm file, nhỏ gọn)
# False => toàn bộ slice từ tất cả COVID19 zips (rất lớn)
FILTER_LESION_SLICES = True

In [ ]:
# ── Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
os.makedirs(f"{DRIVE_OUTPUT}/images", exist_ok=True)
print(f"Output => {DRIVE_OUTPUT}")

In [ ]:
# ── Tải CSV + COVID19-*.zip, giải nén, lưu vào Drive ──────────────────
import requests, zipfile, csv, shutil, time
from pathlib import Path

BASE_URL = "https://download.cncb.ac.cn/covid-ct"
OUT      = Path(DRIVE_OUTPUT)
TMP      = Path("/content/_ccccii_tmp")
TMP.mkdir(exist_ok=True)

# ── Helper: tải với resume + retry ────────────────────────────────────
def download(url, dst, max_retries=5):
    dst = Path(dst)
    if dst.exists() and dst.stat().st_size > 0:
        print(f"  [skip] {dst.name}")
        return
    part = dst.with_suffix(dst.suffix + ".part")
    for attempt in range(1, max_retries + 1):
        resume_pos = part.stat().st_size if part.exists() else 0
        headers    = {"Range": f"bytes={resume_pos}-"} if resume_pos > 0 else {}
        try:
            with requests.get(url, headers=headers, stream=True, timeout=60) as r:
                r.raise_for_status()
                total = int(r.headers.get("Content-Length", 0)) + resume_pos
                mode  = "ab" if resume_pos > 0 else "wb"
                t0    = time.time()
                with open(part, mode) as f:
                    done = resume_pos
                    for chunk in r.iter_content(chunk_size=1 << 20):  # 1 MB chunks
                        f.write(chunk)
                        done += len(chunk)
                        if total > 0:
                            pct = min(done / total * 100, 100)
                            bar = int(pct / 5)
                            print(f"\r  {'█'*bar}{'░'*(20-bar)} {pct:5.1f}%  ",
                                  end="", flush=True)
            part.rename(dst)
            elapsed = time.time() - t0
            size_mb = dst.stat().st_size / 1e6
            print(f" {size_mb:.0f} MB  ({elapsed:.0f}s)")
            return
        except Exception as e:
            print(f"\n  [attempt {attempt}/{max_retries}] {type(e).__name__}: {e}")
            if attempt < max_retries:
                wait = 5 * attempt
                print(f"  Thử lại sau {wait}s ...")
                time.sleep(wait)
    raise RuntimeError(f"Tải thất bại sau {max_retries} lần: {url}")

# ── 1. Tải CSV ─────────────────────────────────────────────────────────
for fname in ["lesions_slices.csv", "metadata.csv"]:
    print(f"Tải {fname} ...", end=" ", flush=True)
    download(f"{BASE_URL}/{fname}", OUT / fname)

# ── 2. Đọc lesions_slices.csv - cột imgpath ────────────────────────────
# Định dạng imgpath: COVID19-N/patient_id/scan_id/slice.png
# Key để khớp với file giải nén: (patient_id, scan_id, stem)
lesion_keys = set()
csv_path    = OUT / "lesions_slices.csv"
if FILTER_LESION_SLICES and csv_path.exists():
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        cols   = reader.fieldnames or []
        print(f"\nColumns: {cols}")
        for row in reader:
            imgpath = (row.get("imgpath") or "").strip().replace("\\", "/")
            if not imgpath:
                continue
            parts = imgpath.rstrip("/").split("/")
            # Lấy 3 thành phần cuối: patient_id / scan_id / filename
            if len(parts) >= 3:
                patient_id  = parts[-3]
                scan_id     = parts[-2]
                slice_stem  = Path(parts[-1]).stem   # bỏ .png
                lesion_keys.add((patient_id, scan_id, slice_stem))
            elif len(parts) == 2:
                lesion_keys.add((parts[-2], parts[-1], ""))  # fallback
    print(f"Lọc: {len(lesion_keys)} slice có tổn thương")
    # Debug: in 3 mẫu key
    for k in list(lesion_keys)[:3]:
        print(f"  mẫu key: {k}")
else:
    print("Không lọc - giữ toàn bộ slice")

# ── 3. Tải + giải nén từng COVID19-N.zip ──────────────────────────────
parts_list  = list(range(1, 32)) if DOWNLOAD_ALL else TEST_PARTS
img_out     = OUT / "images"
total_copied = 0

for idx, i in enumerate(parts_list, 1):
    fname    = f"COVID19-{i}.zip"
    zip_path = TMP / fname
    print(f"\n[{idx}/{len(parts_list)}] {fname}")
    download(f"{BASE_URL}/{fname}", zip_path)

    extract_dir = TMP / f"ext_{i}"
    extract_dir.mkdir(exist_ok=True)
    print(f"  Giải nén ...", end="", flush=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    print(" xong")

    # Cấu trúc sau giải nén: extract_dir/COVID19-N/<patient>/<scan>/<slice>.png
    copied = 0
    for png in sorted(extract_dir.rglob("*.png")):
        p = png.parts
        # Lấy 3 phần cuối của path (patient, scan, file)
        if len(p) >= 3:
            patient_id  = p[-3]
            scan_id     = p[-2]
            slice_stem  = png.stem
            key = (patient_id, scan_id, slice_stem)
        else:
            key = None

        if lesion_keys and key not in lesion_keys:
            continue

        out_name = f"{p[-3]}_{p[-2]}_{png.name}" if len(p) >= 3 else png.name
        dst = img_out / out_name
        if not dst.exists():
            shutil.copy2(png, dst)
            copied += 1

    total_copied += copied
    print(f"  => {copied} slice copied  (tổng: {total_copied})")

    # Dọn dẹp ngay để tiết kiệm disk Colab
    shutil.rmtree(extract_dir, ignore_errors=True)
    zip_path.unlink(missing_ok=True)

print(f"\n{'='*50}")
print(f"Hoàn tất: {total_copied} CT slices => {img_out}")

In [ ]:
# ── Debug: In 5 dòng đầu lesions_slices.csv ───────────────────────────
# Chạy cell này nếu tổng copied = 0 để kiểm tra format thực tế của CSV
import csv
from pathlib import Path

csv_path = Path(DRIVE_OUTPUT) / "lesions_slices.csv"
with open(csv_path, newline="") as f:
    reader = csv.DictReader(f)
    print(f"Columns: {reader.fieldnames}")
    print()
    for i, row in enumerate(reader):
        print(dict(row))
        if i >= 4: break

# In 3 tên file PNG thực tế trong extract để so sánh
from pathlib import Path
tmp = Path("/content/_ccccii_tmp")
sample_pngs = list(tmp.rglob("*.png"))[:3]
if sample_pngs:
    print("\nMẫu path PNG trong zip:")
    for p in sample_pngs:
        print(" ", p)
else:
    print("\n(Không còn file trong tmp - zip đã bị xóa, cần tải lại 1 zip để debug)")

In [ ]:
# ── Kiểm tra kết quả ──────────────────────────────────────────────────
from pathlib import Path

out  = Path(DRIVE_OUTPUT)
imgs = list((out / "images").glob("*.png"))

print(f"{'='*50}")
print(f"  CC-CCII => {out}")
print(f"{'='*50}")
print(f"  images/          {len(imgs):>6} files")
for fname in ["lesions_slices.csv", "metadata.csv"]:
    f = out / fname
    size = f"{f.stat().st_size/1024:.0f} KB" if f.exists() else "MISSING"
    print(f"  {fname:<24} {size}")

if len(imgs) > 0:
    print(f"\n  Mẫu tên file: {imgs[0].name}")
    print("  Dataset sẵn sàng.")

In [ ]:
# ── Xem thử 6 CT slice ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

imgs = sorted(Path(DRIVE_OUTPUT, "images").glob("*.png"))[:6]
if not imgs:
    print("Không có ảnh để hiển thị.")
else:
    fig, axes = plt.subplots(1, len(imgs), figsize=(3 * len(imgs), 3))
    if len(imgs) == 1:
        axes = [axes]
    for ax, p in zip(axes, imgs):
        ax.imshow(mpimg.imread(str(p)), cmap="gray")
        ax.set_title(p.name[:20], fontsize=7)
        ax.axis("off")
    plt.suptitle("CC-CCII - COVID-19 CT slices", fontweight="bold")
    plt.tight_layout()
    plt.show()